In [2]:
import pandas as pd
import numpy as np

# Create a Data Ingestion Function

In [3]:
import pandas as pd

def data_ingestion(filename):
    """
    Load the dataset and return it as a pandas DataFrame.
    """

    # Reading the data
    df = pd.read_csv(filename)

    return df

In [4]:
filename = "/content/car_pricing_dataset.csv"
df = data_ingestion(filename)
df.head()

,Car_Age,Mileage_km,Engine_CC,Horsepower,Fuel_Type,Transmission,Brand,Body_Type,Previous_Owners,Car_Price_INR
0,6,110878,3000,128,Petrol,Automatic,Audi,Hatchback,1,3541000
1,3,169899,2500,296,CNG,Manual,Ford,SUV,0,3461000
2,12,67623,800,63,Petrol,Manual,Tata,Hatchback,1,786000
3,14,80450,3000,75,CNG,Manual,Mercedes,Hatchback,0,2520000
4,10,98426,1200,83,Petrol,Automatic,Toyota,MUV,3,1039000


## Create a Data Validation Function

In [5]:
def data_validation(df):
    """
    Validate the dataset before training.
    """

    print("=" * 50)
    print("DATA VALIDATION REPORT")
    print("=" * 50)

    # Shape
    print(f"Dataset Shape : {df.shape}")

    # Missing Values
    print("\nMissing Values:")
    print(df.isnull().sum())

    # Duplicate Rows
    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    # Data Types
    print("\nData Types:")
    print(df.dtypes)

    # Target Column
    if "target" in df.columns:
        print("\nTarget column found ✔")
    else:
        print("\nTarget column missing ❌")

    print("=" * 50)

    return df

In [6]:
filename = "/content/car_pricing_dataset.csv"
df = data_ingestion(filename)
df = data_validation(df)

DATA VALIDATION REPORT
Dataset Shape : (200, 10)

Missing Values:
Car_Age            0
Mileage_km         0
Engine_CC          0
Horsepower         0
Fuel_Type          0
Transmission       0
Brand              0
Body_Type          0
Previous_Owners    0
Car_Price_INR      0
dtype: int64

Duplicate Rows:
0

Data Types:
Car_Age             int64
Mileage_km          int64
Engine_CC           int64
Horsepower          int64
Fuel_Type          object
Transmission       object
Brand              object
Body_Type          object
Previous_Owners     int64
Car_Price_INR       int64
dtype: object

Target column missing ❌


In [7]:
from sklearn.model_selection import train_test_split

def split_data(df):


    y = df[["Car_Price_INR"]]
    X = df.drop("Car_Price_INR", axis=1)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    print("Train Test Split Completed")

    return X_train, X_test, y_train, y_test

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

def data_transformation(X_train, X_test):

    numerical_columns = X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns

    categorical_columns = X_train.select_dtypes(
        include=["object"]
    ).columns

    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

    X_train = preprocessor.fit_transform(X_train)

    X_test = preprocessor.transform(X_test)

    print("Data Transformation Completed")

    return X_train, X_test, preprocessor

In [9]:
filename = "/content/car_pricing_dataset.csv"
df = data_ingestion(filename)
df = data_validation(df)
X_train, X_test, y_train, y_test = split_data(df)

X_train, X_test, preprocessor = data_transformation(
    X_train,
    X_test
)


DATA VALIDATION REPORT
Dataset Shape : (200, 10)

Missing Values:
Car_Age            0
Mileage_km         0
Engine_CC          0
Horsepower         0
Fuel_Type          0
Transmission       0
Brand              0
Body_Type          0
Previous_Owners    0
Car_Price_INR      0
dtype: int64

Duplicate Rows:
0

Data Types:
Car_Age             int64
Mileage_km          int64
Engine_CC           int64
Horsepower          int64
Fuel_Type          object
Transmission       object
Brand              object
Body_Type          object
Previous_Owners     int64
Car_Price_INR       int64
dtype: object

Target column missing ❌
Train Test Split Completed
Data Transformation Completed


## Create Model Training Function

In [11]:
from sklearn.tree import DecisionTreeRegressor

def model_training(X_train, y_train):
    """
    Train the Decision Tree model.
    """

    # Create model
    model = DecisionTreeRegressor(
        max_depth=3,
        random_state=42
    )

    # Train model
    model.fit(X_train, y_train)

    print("="*50)
    print("MODEL TRAINING COMPLETED")
    print("="*50)

    return model

In [12]:
model = model_training(X_train, y_train)

MODEL TRAINING COMPLETED


In [15]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

def model_evaluation(model, X_test, y_test):
    """
    Evaluate the trained model.
    """
    # Prediction
    y_pred = model.predict(X_test)
    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("=" * 50)
    print("MODEL EVALUATION")
    print("=" * 50)
    return mae,r2

In [18]:
mae,r2 = model_evaluation(
    model,
    X_test,
    y_test
)

print(mae)
print(r2)

MODEL EVALUATION
449045.67555887083
0.49878211558291974
